# Weeks 11–12 — Qwen3.5-4B QLoRA recovery (Colab)

Completes the 4-bit recovery arm that **did not fit the 8 GB laptop** (`recovery_week11-12.md` §5).

**What this notebook does**
1. Clones `herai/week_11_12` and installs dependencies
2. Uploads the existing 4B measurement artifact (gitignored locally)
3. Runs `scripts/run_recovery.py` with the Colab-sized QLoRA config
4. Shows the table and lets you download the JSON + figure

**Runtime**
- Runtime → Change runtime type → **T4 GPU** (16 GB) is enough; L4/A100 also fine
- Wall time: roughly **45–90 min** on a T4 (2 model loads × eval + 2×100 LoRA steps + PIQA)
- Free Colab can disconnect — download the zip as soon as the run finishes

**Open from GitHub:** after this notebook is on the branch, use  
`https://colab.research.google.com/github/geeeeenccc/Redunformer-Neuron/blob/main/notebooks/group_5/week11_12_qlora_4b_colab.ipynb`


## 0. GPU check

Stop here if you see no GPU — the QLoRA run needs one.

In [ ]:
!nvidia-smi
import torch
assert torch.cuda.is_available(), "Enable a GPU runtime: Runtime → Change runtime type → T4 GPU"
print(f"cuda: {torch.cuda.get_device_name(0)}")
print(f"vram: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

## 1. Clone the branch and install

Set `REPO_URL` if you forked the repo. A public clone is enough — no GitHub token needed for this project.

In [ ]:
import os
from pathlib import Path

REPO_URL = "https://github.com/geeeeenccc/Redunformer-Neuron.git"
BRANCH = "main"
REPO_DIR = Path("/content/Redunformer-Neuron")

if not (REPO_DIR / ".git").exists():
    !git clone --branch {BRANCH} --depth 1 {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin {BRANCH}
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull --ff-only origin {BRANCH}

os.chdir(REPO_DIR)
%cd {REPO_DIR}
!pip install -q -e "." 2>&1 | tail -n 5

print("cwd:", Path.cwd())
print("recovery configs:")
for f in sorted(Path("configs/recovery").glob("*.yaml")):
    print(" ", f)
assert Path("configs/recovery/lora_smoke_4b.yaml").exists(), (
    "Clone is missing lora_smoke_4b.yaml ? Runtime ? Disconnect and delete runtime, "
    "then re-run from the clone cell so you get a fresh pull of herai/week_11_12."
)


## 2. (Optional) Hugging Face token

Only needed if Hub rate-limits you or the model becomes gated. Leave blank otherwise.

In [ ]:
import os
from getpass import getpass

token = os.environ.get("HF_TOKEN") or getpass("HF token (Enter to skip): ").strip()
if token:
    os.environ["HF_TOKEN"] = token
    os.environ["HUGGING_FACE_HUB_TOKEN"] = token
    from huggingface_hub import login
    login(token=token, add_to_git_credential=False)
    print("logged in")
else:
    print("no token — using anonymous Hub access")

## 3. Bring in the 4B measurement artifact

The ranking JSON + `.npz` live under `experiments/results/` and are **gitignored**, so Colab does not get them from the clone.

Upload these two files from your laptop (same folder):

- `measurement_Qwen__Qwen3.5-4B_20260629T183140Z.json` (~34 KB)
- `measurement_Qwen__Qwen3.5-4B_20260629T183140Z.npz` (~8.5 MB)

Path on your machine: `experiments/results/` inside the repo.

If you skip the upload, cell 3b re-runs the Weeks 5–8 measurement on Colab instead (~10–20 min).

In [ ]:
from pathlib import Path
import os

REPO_DIR = Path("/content/Redunformer-Neuron")
assert (REPO_DIR / "scripts" / "run_recovery.py").exists(), (
    f"Repo not found at {REPO_DIR}. Run the clone cell first."
)
os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

from pathlib import Path
from google.colab import files

RESULTS = Path("experiments/results")
RESULTS.mkdir(parents=True, exist_ok=True)
(RESULTS / "figures").mkdir(parents=True, exist_ok=True)

TARGET_JSON = RESULTS / "measurement_Qwen__Qwen3.5-4B_20260629T183140Z.json"
TARGET_NPZ = RESULTS / "measurement_Qwen__Qwen3.5-4B_20260629T183140Z.npz"

if TARGET_JSON.exists() and TARGET_NPZ.exists():
    print("measurement already present:")
    print(" ", TARGET_JSON, TARGET_JSON.stat().st_size)
    print(" ", TARGET_NPZ, TARGET_NPZ.stat().st_size)
else:
    print("Upload the .json and .npz measurement files ?")
    uploaded = files.upload()
    for name, data in uploaded.items():
        dest = RESULTS / Path(name).name
        dest.write_bytes(data)
        print(f"saved {dest} ({len(data):,} bytes)")

assert TARGET_JSON.exists() and TARGET_NPZ.exists(), (
    "Missing measurement files. Re-run this cell and upload both, or run cell 3b."
)
print("ready")


### 3b. Fallback — re-measure on Colab

Only if you could not upload the artifact. Skip if cell 3 already succeeded.

In [ ]:
from pathlib import Path
import os

REPO_DIR = Path("/content/Redunformer-Neuron")
assert (REPO_DIR / "scripts" / "run_recovery.py").exists(), (
    f"Repo not found at {REPO_DIR}. Run the clone cell first."
)
os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

from pathlib import Path
import yaml

TARGET_JSON = Path("experiments/results/measurement_Qwen__Qwen3.5-4B_20260629T183140Z.json")
if TARGET_JSON.exists():
    print("skip ? measurement already present")
else:
    cfg = yaml.safe_load(Path("configs/measurement/neuron_activations.yaml").read_text())
    cfg["model_config"] = "configs/models/qwen3.5-4b_colab.yaml"
    cfg["calibration"]["max_blocks"] = 128
    patch = Path("configs/measurement/_colab_neuron_activations.yaml")
    patch.write_text(yaml.dump(cfg))
    !python scripts/run_measurement.py --config {patch}
    latest = sorted(Path("experiments/results").glob("measurement_Qwen__Qwen3.5-4B_*.json"))[-1]
    print("wrote", latest)
    print("Update configs/recovery/lora_qwen3.5-4b_colab.yaml measurement_path to:", latest)


## 4. Smoke test the 4-bit path (optional, ~5 min)

Confirms adapters train on the quantized model with masks live before spending an hour on the full run. Safe to skip if you are short on Colab quota.

In [ ]:
from pathlib import Path
import os

REPO_DIR = Path("/content/Redunformer-Neuron")
assert (REPO_DIR / "scripts" / "run_recovery.py").exists(), (
    f"Repo not found at {REPO_DIR}. Run the clone cell first."
)
os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

from pathlib import Path
import yaml

# Prefer the committed smoke config; if the clone is somehow missing it,
# write an equivalent inline so this cell never depends on cwd luck.
smoke_path = Path("configs/recovery/lora_smoke_4b.yaml")
if smoke_path.exists():
    smoke = yaml.safe_load(smoke_path.read_text())
else:
    print("warning: lora_smoke_4b.yaml missing from clone ? using inline defaults")
    smoke = {
        "seed": 42,
        "model_config": "configs/models/qwen3.5-4b_colab.yaml",
        "dataset_config": "configs/datasets/wikitext2.yaml",
        "max_samples": 8,
        "measurement_path": "auto",
        "strategy": "importance",
        "layers": "all",
        "ratios": [0.10],
        "train_unmasked_control": False,
        "train_data": {
            "dataset_config": "configs/datasets/wikitext2_calib.yaml",
            "seq_len": 256,
            "num_blocks": 8,
        },
        "recovery": {
            "rank": 8,
            "lora_alpha": 16,
            "learning_rate": 2.0e-4,
            "steps": 3,
            "batch_size": 1,
            "grad_accum": 1,
            "warmup_steps": 1,
            "gradient_checkpointing": True,
        },
        "eval": {
            "max_length": 1024,
            "stride": 512,
            "run_lm_eval": False,
        },
        "output_dir": "experiments/results",
        "figures_dir": "experiments/results/figures",
        "use_fallback_model": False,
    }

smoke["model_config"] = "configs/models/qwen3.5-4b_colab.yaml"
meas = Path("experiments/results/measurement_Qwen__Qwen3.5-4B_20260629T183140Z.json")
if meas.exists():
    smoke["measurement_path"] = str(meas)
out = Path("configs/recovery/_colab_smoke_4b.yaml")
out.parent.mkdir(parents=True, exist_ok=True)
out.write_text(yaml.dump(smoke))
print("wrote", out.resolve())
print("exists smoke template:", smoke_path.exists(), smoke_path.resolve())

!python scripts/run_recovery.py --config {out} --tag colab_smoke


## 5. Full QLoRA recovery run

Arms: untrained baseline → masked @ 10% → masked+LoRA (100 steps) → unmasked+LoRA control.
Expect **45–90 min** on a T4. Do not close the tab.

In [ ]:
from pathlib import Path
import os

REPO_DIR = Path("/content/Redunformer-Neuron")
assert (REPO_DIR / "scripts" / "run_recovery.py").exists(), (
    f"Repo not found at {REPO_DIR}. Run the clone cell first."
)
os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

!python scripts/run_recovery.py --config configs/recovery/lora_qwen3.5-4b_colab.yaml --tag colab


## 6. Inspect and download results

In [ ]:
from pathlib import Path
import os

REPO_DIR = Path("/content/Redunformer-Neuron")
assert (REPO_DIR / "scripts" / "run_recovery.py").exists(), (
    f"Repo not found at {REPO_DIR}. Run the clone cell first."
)
os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

import json
from pathlib import Path
from IPython.display import Image, display

results = sorted(Path("experiments/results").glob("recovery_colab_*.json"))
assert results, "no recovery_colab_*.json found ? did the run finish?"
out = results[-1]
payload = json.loads(out.read_text())

baseline = payload["baseline"]["perplexity"]
control = payload.get("unmasked_control") or {}
print(f"artifact: {out}")
print(f"baseline PPL: {baseline:.4f}")
if control:
    print(f"unmasked+LoRA control: {control['perplexity']:.4f}")
print()
print(f"{'ratio':>6} {'masked':>10} {'+LoRA':>10} {'raw':>8} {'adj':>8} {'vs ctrl':>10}")
for row in payload["runs"]:
    raw = row.get("recovery_fraction")
    adj = row.get("adjusted_recovery_fraction")
    residual = row.get("residual_vs_control")
    print(
        f"{row['ratio']:>6.0%} "
        f"{row['masked']['perplexity']:>10.4f} "
        f"{row['masked_recovered']['perplexity']:>10.4f} "
        f"{(f'{raw:.1%}' if raw is not None else '-'):>8} "
        f"{(f'{adj:.1%}' if adj is not None else '-'):>8} "
        f"{(f'{residual:+.4f}' if residual is not None else '-'):>10}"
    )

fig = Path(payload["artifacts"]["figures"]["recovery_ppl"])
if fig.exists():
    display(Image(filename=str(fig)))


In [ ]:
from pathlib import Path
import os

REPO_DIR = Path("/content/Redunformer-Neuron")
assert (REPO_DIR / "scripts" / "run_recovery.py").exists(), (
    f"Repo not found at {REPO_DIR}. Run the clone cell first."
)
os.chdir(REPO_DIR)
print("cwd:", Path.cwd())

import shutil
from pathlib import Path
from google.colab import files

bundle = Path("/content/recovery_4b_colab")
if bundle.exists():
    shutil.rmtree(bundle)
bundle.mkdir()

for path in Path("experiments/results").glob("recovery_colab_*"):
    shutil.copy2(path, bundle / path.name)
for path in Path("experiments/results/figures").glob("recovery_ppl_colab_*"):
    shutil.copy2(path, bundle / path.name)

zip_path = shutil.make_archive("/content/recovery_4b_colab", "zip", bundle)
print("downloading", zip_path)
files.download(zip_path)


## 7. Drop results back into the local repo

On your laptop, unzip into `experiments/results/` (JSON) and `experiments/results/figures/` (PNG), then update §5 of `reports/group_5/recovery_week11-12.md` with the Colab numbers.

The headline number to paste is **adjusted recovery** at 10% masking — the same control-adjusted metric reported for the 0.6B run.